In [4]:
import pandas as pd
import folium
from folium.features import DivIcon
import math

# 1. Wastewater Treatment Plant Data (California PoC)
# Flow data approximate (MGD)
plants_data = [
    {"Name": "Hyperion (LA)", "Lat": 33.9250, "Lon": -118.4280, "Flow_MGD": 260},
    {"Name": "San Jose-Santa Clara", "Lat": 37.4260, "Lon": -121.9390, "Flow_MGD": 110},
    {"Name": "Sacramento (EchoWater)", "Lat": 38.4350, "Lon": -121.4980, "Flow_MGD": 130}
]
df_plants = pd.DataFrame(plants_data)

# 2. Theoretical Mineral Supply (Major CA Limestone Quarry Hub)
quarry = {"Name": "Victorville Limestone Hub", "Lat": 34.614, "Lon": -117.216}

# 3. Model Parameters
CO2_TONS_PER_MGD_YEAR = 60 # Based on 100 mg/L CaCO3 dose
CARBON_PRICE = 100 # $/ton (Frontier target)

# 4. Haversine function to calculate supply logistics distance
def calc_distance(lat1, lon1, lat2, lon2):
    R = 3958.8 # Earth radius in miles
    dlat, dlon = math.radians(lat2 - lat1), math.radians(lon2 - lon1)
    a = math.sin(dlat/2)**2 + math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin(dlon/2)**2
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

# 5. Perform the Analysis
df_plants['Annual_CO2_Removal_Tons'] = df_plants['Flow_MGD'] * CO2_TONS_PER_MGD_YEAR
df_plants['Annual_Carbon_Revenue_$'] = df_plants['Annual_CO2_Removal_Tons'] * CARBON_PRICE
df_plants['Miles_to_Quarry'] = df_plants.apply(lambda row: calc_distance(row['Lat'], row['Lon'], quarry['Lat'], quarry['Lon']), axis=1)

print(df_plants[['Name', 'Flow_MGD', 'Annual_CO2_Removal_Tons', 'Miles_to_Quarry']])
print(f"\nTotal Annual CO2 Potential (3 Plants): {df_plants['Annual_CO2_Removal_Tons'].sum():,} tons")

# 6. Generate the Web Map
#m = folium.Map(location=[36.5, -119.5], zoom_start=6, tiles="CartoDB dark_matter")
m = folium.Map(
    location=[36.5, -119.5],
    zoom_start=6,
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Dark_Gray_Base/MapServer/tile/{z}/{y}/{x}',
    attr='Tiles &copy; Esri &mdash; Esri, DeLorme, NAVTEQ'
)
folium.Marker([quarry['Lat'], quarry['Lon']], popup=quarry['Name'], icon=folium.Icon(color='gray', icon='industry', prefix='fa')).add_to(m)

for idx, row in df_plants.iterrows():
    folium.CircleMarker(
        location=[row['Lat'], row['Lon']],
        radius=row['Flow_MGD'] / 15, # Scale bubble by flow
        color='cyan', fill=True, fill_color='cyan',
        popup=f"{row['Name']}<br>Flow: {row['Flow_MGD']} MGD<br>CO2: {row['Annual_CO2_Removal_Tons']:,} tons/yr"
    ).add_to(m)

m # Displays the interactive map in Colab

                     Name  Flow_MGD  Annual_CO2_Removal_Tons  Miles_to_Quarry
0           Hyperion (LA)       260                    15600        83.996139
1    San Jose-Santa Clara       110                     6600       327.665198
2  Sacramento (EchoWater)       130                     7800       355.194269

Total Annual CO2 Potential (3 Plants): 30,000 tons


In [5]:
import pandas as pd
import geopandas as gpd
import folium

# 1. Ingest Raw CSVs (Replace these actual Colab file paths)
# df_epa = pd.read_csv('epa_echo_ca_flows.csv')
# df_usgs = pd.read_csv('usgs_ca_limestone.csv')

# --- TEMPORARY MOCK DATA TO TEST THE PIPELINE BEFORE UPLOAD ---
df_epa = pd.DataFrame({
    "Facility_Name": ["Hyperion", "San Jose", "Sacramento", "Fresno-Clovis", "Bakersfield"],
    "Lat": [33.9250, 37.4260, 38.4350, 36.7323, 35.3211],
    "Lon": [-118.4280, -121.9390, -121.4980, -120.0253, -118.9839],
    "Flow_MGD": [260, 110, 130, 68, 25]
})

df_usgs = pd.DataFrame({
    "Quarry_Name": ["Victorville Hub", "Cupertino Quarry", "Lucerne Valley"],
    "Lat": [34.614, 37.318, 34.439],
    "Lon": [-117.216, -122.093, -116.942],
    "Status": ["Active", "Active", "Active"]
})
# --------------------------------------------------------------

# 2. Convert to GeoDataFrames (Standard WGS84 EPSG:4326)
gdf_plants = gpd.GeoDataFrame(df_epa, geometry=gpd.points_from_xy(df_epa.Lon, df_epa.Lat), crs="EPSG:4326")
gdf_quarries = gpd.GeoDataFrame(df_usgs, geometry=gpd.points_from_xy(df_usgs.Lon, df_usgs.Lat), crs="EPSG:4326")

# 3. Reproject to California Albers (EPSG:3310) for accurate distance measurement in meters
gdf_plants_proj = gdf_plants.to_crs("EPSG:3310")
gdf_quarries_proj = gdf_quarries.to_crs("EPSG:3310")

# 4. Spatial Join: Find the nearest quarry for every WWTP
# Calculates distance in meters, then we convert to miles
merged_gdf = gpd.sjoin_nearest(gdf_plants_proj, gdf_quarries_proj, distance_col="Distance_Meters")
merged_gdf['Miles_to_Nearest_Quarry'] = merged_gdf['Distance_Meters'] * 0.000621371

# 5. Apply the Chemistry & Economic Logic
CO2_TONS_PER_MGD_YEAR = 60
merged_gdf['Annual_CO2_Removal_Tons'] = merged_gdf['Flow_MGD'] * CO2_TONS_PER_MGD_YEAR

# Reproject back to standard GPS coordinates for web mapping
final_gdf = merged_gdf.to_crs("EPSG:4326")

# 6. Generate the Map (Matching the Esri Dark Gray aesthetic)
m = folium.Map(
    location=[36.5, -119.5],
    zoom_start=6,
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Dark_Gray_Base/MapServer/tile/{z}/{y}/{x}',
    attr='Tiles &copy; Esri'
)

# Plot all active quarries
for idx, row in gdf_quarries.iterrows():
    folium.Marker(
        [row['Lat'], row['Lon']],
        popup=row['Quarry_Name'],
        icon=folium.Icon(color='gray', icon='industry', prefix='fa')
    ).add_to(m)

# Plot wastewater plants scaled by flow
for idx, row in final_gdf.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=row['Flow_MGD'] / 15,
        color='cyan', fill=True, fill_color='cyan',
        popup=f"{row['Facility_Name']}<br>Flow: {row['Flow_MGD']} MGD<br>CO2: {row['Annual_CO2_Removal_Tons']:,} tons/yr<br>Nearest Quarry: {row['Quarry_Name']} ({row['Miles_to_Nearest_Quarry']:.1f} mi)"
    ).add_to(m)

m

In [7]:
# Export the final GeoDataFrame to a generic GeoJSON
final_gdf.to_file("ca_wwtp_cdr_viability.geojson", driver="GeoJSON")
print("Export complete: ca_wwtp_cdr_viability.geojson")

Export complete: ca_wwtp_cdr_viability.geojson
